In [1]:
import os
import json
import re
import pandas as pd

pd.options.display.max_columns = None

In [2]:
def extract_json(response: str):
    """Extract JSON content from a formatted string."""
    match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    if match:
        json_str = match.group(1)
    else:
        json_str = response.strip('```json').strip('```')
    
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Chyba při dekódování JSON: {e}")
        return None

In [3]:
def process_txt_files(folder_path, prefix):
    """Zpracuje všechny txt soubory začínající prefixem (např. 'bank_part') ve složce a vrátí Pandas DataFrame."""
    all_data = []
    
    # Get files with prefix and end with .txt
    files = [f for f in os.listdir(folder_path) if f.startswith(prefix) and f.endswith(".txt")]
    
    # Order by number
    files.sort(key=lambda x: int(re.search(r'chunk(\d+)', x).group(1)))
    
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as file:
            for line in file:
                try:
                    json_obj = json.loads(line.strip())
                    content_str = json_obj.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
                    extracted_json = extract_json(content_str)
                    
                    if extracted_json:
                        row = {"id": json_obj["id"], "custom_id": json_obj["custom_id"]}
                        for feature in extracted_json.get("features", []):
                            row[feature["feature_name"]] = feature["answer"]
                        
                        all_data.append(row)
                except json.JSONDecodeError:
                    print(f"Chyba dekódování JSON v souboru {filename}")

    df = pd.DataFrame(all_data)
    return df

In [4]:
# Použití skriptu
folder_path = "./"
df = process_txt_files(folder_path, "bank_")

# Zobrazení výsledného dataframe
df

,id,custom_id,transaction_type,issue_type,card_status,currency_involved,verification_requirement,transaction_status,payment_method,customer_action,service_availability,time_reference,charge_type,security_concern,account_action,support_request,card_feature,geographical_reference,communication_channel,frequency_reference,amount_reference,device_reference
0,batch_req_67bbbb4a025c8190918870a865772154,0,card_payment,pending_transaction,active,Other,None,pending,card,None,not_available,None,None,None,None,None,None,None,None,None,None,None
1,batch_req_67bbbb4a14788190840e2b2ec2223fa0,1,card_payment,None,None,Other,None,pending,card,order_card,None,specific_time_frame,None,None,None,None,None,None,None,None,None,None
2,batch_req_67bbbb4a248c8190b13428e2821b2da0,2,card_payment,pending_transaction,active,Other,None,pending,card,None,not_available,specific_time_frame,None,None,None,None,None,None,None,None,None,None
3,batch_req_67bbbb4a34c881909b3c1cead242394b,3,None,None,None,Other,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,batch_req_67bbbb4a44388190bbaf14976a246c93,4,None,lost_or_stolen,lost,Other,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2829,batch_req_67bbeae6caec81908f118aef14797949,2892,None,None,None,Other,None,None,None,None,supported,None,None,None,None,None,None,None,None,None,None,None
2830,batch_req_67bbeae6e908819092b649f8819b7211,2893,None,None,None,None,None,None,None,None,supported,None,None,None,None,None,None,None,None,None,None,None
2831,batch_req_67bbeae7131c8190b1cd07c94d314edc,2894,top_up,None,None,Other,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2832,batch_req_67bbeae7347c819095c68b2c131c228f,2895,top_up,None,active,USD,None,None,card,None,available,None,None,None,None,None,None,country_support,None,None,None,None


# Merge with target

In [5]:
from datasets import load_dataset

data_load = load_dataset("banking77")

In [6]:
load_dataset = pd.DataFrame(data_load['train'])
load_dataset_test = pd.DataFrame(data_load['test'])
load_dataset = pd.concat([load_dataset, load_dataset_test], axis=0)
df['label'] = list(load_dataset['label'])[:len(df)]

In [7]:
df

,id,custom_id,transaction_type,issue_type,card_status,currency_involved,verification_requirement,transaction_status,payment_method,customer_action,service_availability,time_reference,charge_type,security_concern,account_action,support_request,card_feature,geographical_reference,communication_channel,frequency_reference,amount_reference,device_reference,label
0,batch_req_67bbbb4a025c8190918870a865772154,0,card_payment,pending_transaction,active,Other,None,pending,card,None,not_available,None,None,None,None,None,None,None,None,None,None,None,11
1,batch_req_67bbbb4a14788190840e2b2ec2223fa0,1,card_payment,None,None,Other,None,pending,card,order_card,None,specific_time_frame,None,None,None,None,None,None,None,None,None,None,11
2,batch_req_67bbbb4a248c8190b13428e2821b2da0,2,card_payment,pending_transaction,active,Other,None,pending,card,None,not_available,specific_time_frame,None,None,None,None,None,None,None,None,None,None,11
3,batch_req_67bbbb4a34c881909b3c1cead242394b,3,None,None,None,Other,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,11
4,batch_req_67bbbb4a44388190bbaf14976a246c93,4,None,lost_or_stolen,lost,Other,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2829,batch_req_67bbeae6caec81908f118aef14797949,2892,None,None,None,Other,None,None,None,None,supported,None,None,None,None,None,None,None,None,None,None,None,66
2830,batch_req_67bbeae6e908819092b649f8819b7211,2893,None,None,None,None,None,None,None,None,supported,None,None,None,None,None,None,None,None,None,None,None,66
2831,batch_req_67bbeae7131c8190b1cd07c94d314edc,2894,top_up,None,None,Other,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,66
2832,batch_req_67bbeae7347c819095c68b2c131c228f,2895,top_up,None,active,USD,None,None,card,None,available,None,None,None,None,None,None,country_support,None,None,None,None,66


In [8]:
df = df.drop(columns=["id", "custom_id"])
data = pd.get_dummies( 
        df, sparse=False, prefix_sep='_'
    )

In [9]:
data

,label,transaction_type_None,transaction_type_Other,transaction_type_card_payment,transaction_type_cash_withdrawal,transaction_type_deposit,transaction_type_exchange,transaction_type_extra_charge,transaction_type_refund,transaction_type_top_up,transaction_type_transfer,issue_type_None,issue_type_card_not_working,issue_type_declined_transaction,issue_type_extra_charge,issue_type_identity_verification,issue_type_lost_or_stolen,issue_type_pending_transaction,card_status_None,card_status_active,card_status_blocked,card_status_broken,card_status_lost,card_status_stolen,currency_involved_AUD,currency_involved_EUR,currency_involved_GBP,currency_involved_None,currency_involved_Other,currency_involved_USD,"currency_involved_USD, GBP",verification_requirement_None,verification_requirement_identity_verification,verification_requirement_source_of_funds_verification,verification_requirement_top_up_verification,transaction_status_None,transaction_status_completed,transaction_status_failed,transaction_status_pending,transaction_status_reverted,payment_method_None,payment_method_bank_transfer,payment_method_card,payment_method_cash,payment_method_cheque,customer_action_None,customer_action_activate_card,customer_action_cancel_transfer,customer_action_card_linking,customer_action_change_PIN,customer_action_edit_details,customer_action_open_account,customer_action_order_card,customer_action_request_refund,customer_action_terminate_account,service_availability_None,service_availability_available,service_availability_not_available,service_availability_supported,time_reference_ASAP,time_reference_None,time_reference_immediate,time_reference_soon,time_reference_specific_time_frame,time_reference_urgent,charge_type_None,charge_type_exchange_rate,charge_type_extra_charge,charge_type_fee,charge_type_hidden_fee,security_concern_None,security_concern_compromised_card,security_concern_lost_phone,account_action_None,account_action_cancel_request,account_action_edit_personal_details,account_action_terminate_account,support_request_ATM_support,support_request_None,support_request_card_linking,card_feature_None,card_feature_contactless,geographical_reference_None,geographical_reference_country_support,geographical_reference_currency,communication_channel_None,communication_channel_app_notification,communication_channel_email,frequency_reference_None,frequency_reference_multiple_times,frequency_reference_once,amount_reference_None,amount_reference_specific_amount,amount_reference_wrong_amount,device_reference_ATM,device_reference_None,device_reference_app,device_reference_phone
0,11,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,True,False,True,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False
1,11,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,True,False,False,False,False,True,False,False,True,False,False,False,False,True,False,True,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False
2,11,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,F

In [10]:
# 1) Libraries
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 2) Příprava feature matic X a cílové proměnné y
X = data.drop(columns=["label"])
y = data["label"]
#categorical_columns = X.select_dtypes(include=["object"]).columns
#X = df.drop(columns=categorical_columns)

# 3) Rozdělení na trénovací a testovací sadu
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)  # stratify, pokud je to klasifikace s nerovnoměrnými třídami

# ----------------------------------------------------------------
# 4) Definice modelu RandomForestClassifier
model = RandomForestClassifier(random_state=42)

# 5) Nastavení rozsahu parametrů pro RandomizedSearchCV
param_dist = {
    "n_estimators": [50, 100, 200],       # Počet stromů v lese
    "max_depth": [3, 5, 10, None],        # Maximální hloubka stromu
    "min_samples_split": [2, 5, 10],      # Minimální počet vzorků pro split
    "min_samples_leaf": [1, 2, 5],        # Minimální počet vzorků v listu
}

# 6) Konfigurace RandomizedSearchCV (n_iter a cv lze upravit dle potřeby)
random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,             # kolik náhodných kombinací parametrů prozkoumat
    cv=5,                  # 5-fold cross-validace
    scoring="accuracy",    # metrika, dle které se bude model porovnávat
    random_state=42,
    n_jobs=-1,             # využití všech CPU jader pro rychlejší výpočet
    verbose=1
)

# 7) Trénink modelu s vyhledáváním nejlepších hyperparametrů
random_search.fit(X_train, y_train)

# 8) Vypsání nejlepších parametrů a skóre
print("Nejlepší parametry:", random_search.best_params_)
print("Nejlepší skóre na trénovací cross-validaci:", random_search.best_score_)

# 9) Ověření na testovací sadě
best_model = random_search.best_estimator_  # získáme nejlepší nalezený model
y_pred = best_model.predict(X_test)

# 10) Vyhodnocení
print("Přesnost na testu:", accuracy_score(y_test, y_pred))
print("Classification report na testu:")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Nejlepší parametry: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 10}
Nejlepší skóre na trénovací cross-validaci: 0.7759167955188612
Přesnost na testu: 0.7495590828924162
Classification report na testu:
              precision    recall  f1-score   support

           1       0.34      0.95      0.51        22
           4       0.62      0.64      0.63        25
           8       0.74      0.72      0.73        32
          11       0.64      0.52      0.57        31
          12       0.93      0.59      0.72        22
          13       0.95      0.75      0.84        28
          14       0.87      0.91      0.89        22
          15       0.76      0.68      0.72        38
          17       0.97      0.91      0.94        34
          23       1.00      0.71      0.83         7
          32       0.46      0.86      0.60        22
          33       0.69      0.38      0.49       

In [11]:
grid_rfc = {
 'max_depth': [10, 20, 30, 40, 45, 50],
 'max_features': ['log2', 'sqrt', 30],
 'min_samples_leaf': [30, 40, 50, 60, 70, 90],
 'min_samples_split': [20, 30, 50, 60, 90],
 'n_estimators': [ 100, 200, 400, 1000]}

In [12]:
def train_gbc(X_train, y_train, X_test, y_test, metric = None):
    gbc = GradientBoostingClassifier()
    gbc_grid = RandomizedSearchCV(estimator = gbc, n_iter=2000 , param_distributions = grid_rfc, cv = 5, verbose=3, n_jobs = -1, scoring=metric, random_state=42)
    gbc_grid.fit(X_train, y_train)
    best = gbc_grid.best_estimator_

    y_pred = gbc_grid.predict(X_test)
    display(f"Best params: {gbc_grid.best_params_}")
    display(f"Gradient Boosting Classifier Accuracy score: {accuracy_score(y_test, y_pred)}")
    display(f"Gradient Boosting Classifier Recall score: {recall_score(y_test, y_pred, average='macro')}")
    display(f"Gradient Boosting Classifier F1 score: {f1_score(y_test,  y_pred, average='macro')}")
    display(f"Gradient Boosting Classifier Precision score: {precision_score(y_test, y_pred, average='macro')}")
    display(f"Gradient Boosting Classifier MAE score: {mean_absolute_error(y_test, y_pred)}")
    display(f"Gradient Boosting Classifier RMSE score: {root_mean_squared_error(y_test, y_pred)}")
    display(f"Gradient Boosting Classifier MSE score: {mean_squared_error(y_test, y_pred)}")
    get_conf_matrix(y_test, y_pred)


    result = permutation_importance(
        best, X_test, y_test, n_repeats=30, random_state=42, n_jobs=2
    )
    forest_importances = pd.Series(result.importances_mean, index=X_train.columns)
    forest_importances = forest_importances.sort_values(ascending=False)[:16]
    display(forest_importances)

    fig, ax = plt.subplots()
    forest_importances.plot.bar(ax=ax)
    ax.set_title("Feature importances using permutation on full model")
    ax.set_ylabel("Mean accuracy decrease")
    fig.tight_layout()
    plt.show()

    # Compute SHAP values
    explainer = shap.Explainer(best, X_train)
    shap_values = explainer(X_test)

    # Plot SHAP summary plot
    shap.summary_plot(shap_values, X_test)